In [ ]:
import matplotlib.pyplot as plt 
import pandas as pd 

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from tda_slugging.utils import read_files, find_shortest_file, batch_analyzer, mutual_information

# Data paths (type-3 slugging is committed; other types need the full 3W dataset)
DATA_3W_ALL      = str(REPO_ROOT / 'data' / '3W' / 'ALL')
DATA_3W_REAL     = str(REPO_ROOT / 'data' / '3W' / 'REAL')
DATA_3W_SIM      = str(REPO_ROOT / 'data' / '3W' / 'SIMULATED')
DATA_WELL        = str(REPO_ROOT / 'data' / 'well')
OUTPUTS_DIR      = str(REPO_ROOT / 'outputs')
IMAGES_DIR       = str(REPO_ROOT / 'images')
# Paths below require the full 3W dataset (download from https://github.com/ricardovvargas/3w_dataset):
# DATA_3W_NORMAL   = '/path/to/3w_dataset/data/0'
# DATA_3W_UNSTABLE = '/path/to/3w_dataset/data/4'


In [ ]:
import numpy as np
from ripser import ripser
from persim import plot_diagrams
from gtda.diagrams import PersistenceEntropy, PairwiseDistance
from gtda.diagrams import Silhouette
from gtda.diagrams import BettiCurve 
from gtda.homology import VietorisRipsPersistence
from gtda.time_series import TakensEmbedding, SlidingWindow
import plotly.express as px
from sklearn.metrics import mutual_info_score
from gtda.metaestimators import CollectionTransformer
from gtda.pipeline import Pipeline
from gtda.plotting import plot_heatmap


## TDA of white Noise 

First we test how topological data analysis (TDA) performs on noisey stationary timeseries (structureless). Let's create a white noise timeseries for benchmarking the TDA perfomances.

In [ ]:
from random import gauss
from random import seed
from pandas.plotting import autocorrelation_plot
from statsmodels.graphics.gofplots import qqplot

# seed random number generator
seed(123456)
# create white noise series
white_noise_gen = [gauss(0.0, .50) for i in range(1000)]
white_noise = pd.Series(white_noise_gen)

# I am assuming 1000 points sampled every 10s (sampling frequency 0.1Hz)
# therefore the sampling frequency after the FFT is 0.1/1000 = 1e-4 Hz

fig_wn, data = plt.subplots(1, 3, figsize=(17, 3))

fig_wn.suptitle('white noise')
data[0].set_xlabel('time')
data[0].set_ylabel('signal')
data[1].set_xlabel('freq')
data[1].set_ylabel('power spectrum')
data[2].set_xlabel('signal')
data[2].set_ylabel('count')

fft_wn = np.fft.rfft(white_noise)
fft_wn_abs = np.abs(fft_wn)
power_spectrum_wn = np.square(fft_wn_abs)


data[0].plot(white_noise)
data[1].plot(power_spectrum_wn)
white_noise.hist()


In [ ]:
from scipy import signal

f, Pxx_den = signal.periodogram(white_noise, 100)
plt.semilogy(f, Pxx_den)
plt.ylim([1e-5, 1e-1])
plt.xlabel('frequency [Hz]')
plt.ylabel('PSD [V**2/Hz]')
plt.show()

# the frequency index should be fixed... 

The power spectrum is pretty flat, so it looks good as white noise. Now let's check from the QQ plots if this noise is actially Gaussian.

In [ ]:
qqplot(white_noise, line='s')

In [ ]:
from scipy.stats import anderson

# normality test
result = anderson(white_noise)
print('Statistic: %.3f' % result.statistic)
p = 0
for i in range(len(result.critical_values)):
	sl, cv = result.significance_level[i], result.critical_values[i]
	if result.statistic < result.critical_values[i]:
		print('%.3f: %.3f, data looks normal (fail to reject H0)' % (sl, cv))
	else:
		print('%.3f: %.3f, data does not look normal (reject H0)' % (sl, cv))

And let's check if it is really uncorrelated. (from the autocorrelation plot)

In [ ]:
from pandas.plotting import autocorrelation_plot
from scipy import signal

autocorrelation_plot(white_noise)


As well, the noise is uncorrellated and Gaussian.

Now, we compute the optimal parameters for the Takens' embedding: the time delay, and the embedding dimension. 
These are computed through some euristhics. For the time delay, we pick the delay corresponding to the first
minimum in the mutual information function (ref), while for the dimension we look at the number of false nearest
neighbours. For white noise these should be meaningless, as the embedding is design to reproduce the attractor 
(i.e. information) contained in a timeseries, while noise does not contain any attractor.
The point cloud should then be Gaussian distributed in d-dimensions. 

In [ ]:
from gtda.time_series import SingleTakensEmbedding, takens_embedding_optimal_parameters
from sklearn.decomposition import PCA
from gtda.plotting import plot_point_cloud

print('length of signal to analyze', len(white_noise_gen))

max_time_delay = 50 
max_embedding_dimension = 20
stride = 1

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    white_noise_gen, max_time_delay, max_embedding_dimension, stride=stride
    )

print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")

embedding_dimension = optimal_embedding_dimension
embedding_time_delay = optimal_time_delay
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_wn_embedded = embedder.fit_transform(white_noise_gen)

pca = PCA(n_components=3)
y_wn_embedded_pca = pca.fit_transform(y_wn_embedded)

plot_point_cloud(y_wn_embedded_pca)

In [ ]:
y_wn_embedded.shape

"y_wn_embedded" is the point cluod array (990 points in 6-dimensions) for Gaussian noise.  

In [ ]:
max_time_delay = 125
bins = len(white_noise_gen)/100

X = white_noise_gen
mutual_information_list = []

for time_delay in range(1, max_time_delay + 1):
    mutual_information_list.append(mutual_information(X, time_delay, n_bins=12))
           

plt.plot(range(1,max_time_delay+1),mutual_information_list);
plt.xlabel('delay');
plt.ylabel('mutual information');

Good, there is no really a minimum in the mutual information of the timeseries, because this is only noise and 
there is no actual information hidden in it.

In this case, the embedding dimension and the optimal time delay are arbitraty. 

In [ ]:
from gtda.homology import VietorisRipsPersistence
homology_dimensions = (0, 1, 2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)


y_wn_embedded_reshaped = y_wn_embedded.reshape(1, *y_wn_embedded.shape)
print(f"y_wn_embedded_reshaped.shape",y_wn_embedded_reshaped.shape)
print(f"y_wn_embedded.shape", y_wn_embedded.shape)


PerHom_wn = VRP.fit_transform(y_wn_embedded_reshaped)
VRP.plot(PerHom_wn)

In [ ]:
pers_H0 = []
pers_H1 = []
max_pers_H0 = []
max_pers_H1 = []

for PersDiag in PerHom_wn: 
    for point in PersDiag:
        birth = point[0]
        death = point[1]
        dimension = point[2]
        persistence = abs(death - birth)
        pers_H0.extend([persistence] if dimension == 0 else [])
        pers_H1.extend([persistence] if dimension == 1 else [])
    max_pers_H0.append(np.amax(pers_H0))
    max_pers_H1.append(np.amax(pers_H1))
    pers_H0 = []
    pers_H1 = []
           
print('Maximum Persistence for BHP:')
print("Max Pers in homology dimension H_0", max_pers_H0 )
print("Max Pers in homology dimension H_1", max_pers_H1 )

As expected the persistence diagram has 0-dimasional components that are Gaussian distributed, and higher 
dimensionality points are just representing noise.

In [ ]:
from ripser import ripser
from persim import plot_diagrams

dgms = ripser(y_wn_embedded, maxdim=2)['dgms']
plot_diagrams(dgms, show=True)

The Takens embedding of a white noise (univariate) time-series gives rise to a n-dimensional ball. The TDA of this object is mostly consisting of points of low persistence, lying close to the diagonal in the persistent homology diagram. 
The the TDA of the n-dimensional ball is therefore a representation of noise: more rigorously, one can define a confidence interval for a given dataset corresponding to a band around the diagonal of the persistence diagram.
The points outside the band are then considered as significant topological features. The width of such confidence band can be estimated rigorously (see Fasy et al. "confidence sets for persistence diagrams", Ann. Statistics (2014) for more details). Other approaches to separate noise from significant data are e.g. the widest gap from the diagonal, from V. Kurlin "A fast persistence-based segmentation of noisy 2D clouds with provable guarantees" Pattern Recognition (2015).

In [ ]:
PE_wn = PersistenceEntropy()
features_wn = PE_wn.fit_transform(PerHom_wn)
print('Persistence Entropy of white noise')
print('Entropy homology class H0:', features_wn[0][0])
print('Entropy homology class H1:', features_wn[0][1])
print('Entropy homology class H2:', features_wn[0][2])

Normalized Shannon entropies are computed follwing: A. Myers, E. Munch, and F. A. Khasawneh, *“Persistent Homology of Complex Networks for Dynamic State Detection”*, Phys. Rev. E 100, 022314, (2019)

According to Myers *et al.* (2019) in order to compare entropies with from persistence diagrams with different number of points the persistent entropy should be normalised as follows:
$$ E'(D) = \frac{E(D)}{\log_2 \mathcal{L}(D)} \qquad  \mathcal{L}(D) = \sum_{x \in D} pers(x) = \sum_{x \in D} 
death_x - birth_x$$  

In [ ]:
PE_wn_norm = PersistenceEntropy(normalize=True)
features_wn_norm = PE_wn_norm.fit_transform(PerHom_wn)
print('Persistence Entropy of white noise')
print('Entropy (normalized) homology class H0:', features_wn_norm[0][0])
print('Entropy (normalized) homology class H1:', features_wn_norm[0][1])
print('Entropy (normalized) homology class H2:', features_wn_norm[0][2])

In [ ]:
Betti = BettiCurve()
Betti_wn = Betti.fit_transform_plot(PerHom_wn)

In [ ]:
Betti_wn.shape

### Deep Dive Gaussian Noise

Ok, it looks like an entropy in the range of 9 is associated with white noise. Can we be sure of this? Here generate many time series (500) and compute the entropy to get an average value. 

In [ ]:
seed(654)
data = []
# create 200 white noise time series each of 1000 time steps. 
# the mean is zero and std dev is 1
for i in range(500):
    white_noise_gen = [gauss(0.0, 1.0) for i in range(1000)]
    white_noise = pd.Series(white_noise_gen)
    data.append(white_noise)
data_T = list(map(list, zip(*data)))
white_noise_df = pd.DataFrame.from_records(data_T)

In [ ]:
white_noise_df

In [ ]:
embedding_dimension = 6
embedding_time_delay = 2
stride = 1

embedder = TakensEmbedding(time_delay=embedding_time_delay,
                           dimension=embedding_dimension,
                           stride=stride)
batch_pca =  CollectionTransformer(PCA(n_components=3),n_jobs=-1)
persistence = VietorisRipsPersistence(homology_dimensions=[0, 1], n_jobs=None)
entropy = PersistenceEntropy(nan_fill_value=-10)
steps = [
         ("embedder", embedder),
         ("pca", batch_pca),
         ("persistence", persistence),
         ("entropy", entropy)
        ]
topological_transfomer = Pipeline(steps)

In [ ]:
WN_entropies = topological_transfomer.fit_transform(white_noise_df.T)

In [ ]:
import statistics as stats

e_h0 = []
e_h1 = []
for e in WN_entropies:
    e_h0.append(e[0])
    e_h1.append(e[1])
    
print('average entropy H0', stats.mean(e_h0), '+/-', stats.stdev(e_h0))
print('average entropy H1', stats.mean(e_h1), '+/-', stats.stdev(e_h1))

In [ ]:
entropy_norm = PersistenceEntropy(normalize=True, nan_fill_value=-10)
steps = [
         ("embedder", embedder),
         ("pca", batch_pca),
         ("persistence", persistence),
         ("entropy", entropy_norm)
        ]
topological_transfomer = Pipeline(steps)

In [ ]:
WN_entropies_norm = topological_transfomer.fit_transform(white_noise_df.T)

In [ ]:
e_h0 = []
e_h1 = []
for e in WN_entropies_norm:
    e_h0.append(e[0])
    e_h1.append(e[1])

print('average normalized entropy H0', stats.mean(e_h0), '+/-', stats.stdev(e_h0))
print('average normalized entropy H1', stats.mean(e_h1), '+/-', stats.stdev(e_h1))

In [ ]:
steps = [
         ("embedder", embedder),
         ("pca", batch_pca),
         ("persistence", persistence)
        ]
topological_transfomer = Pipeline(steps)

In [ ]:
X_diagrams_WN = topological_transfomer.fit_transform(white_noise_df.T)

In [ ]:
pers_H0 = []
pers_H1 = []
max_pers_H0 = []
max_pers_H1 = []

for PersDiag in X_diagrams_WN: 
    for point in PersDiag:
        birth = point[0]
        death = point[1]
        dimension = point[2]
        persistence = abs(death - birth)
        pers_H0.extend([persistence] if dimension == 0 else [])
        pers_H1.extend([persistence] if dimension == 1 else [])
    max_pers_H0.append(np.amax(pers_H0))
    max_pers_H1.append(np.amax(pers_H1))
    pers_H0 = []
    pers_H1 = []
           
fig = px.line(title='Maximum Persistence for BHP, indexed by sliding window number, window of 450 timesteps')
fig.add_scatter(y=max_pers_H0, name="Max Pers in homology dimension H_0")
fig.add_scatter(y=max_pers_H1, name="Max Pers in homology dimension H_1")
fig.show() 

In [ ]:
print("Max Pers in homology dimension H_0", np.amax(max_pers_H0))
print("Max Pers in homology dimension H_1", np.amax(max_pers_H1))

Yes, the values above give an estimate of the entropy for noise: about 9.75 for H$_0$ (1.19 normalized) and 8.45 
for H$_1$ (1.63 normalized). 

## Steady State Flow

Let's analyse now a slugging event.

In [ ]:
slug_1_df = pd.read_csv('slug_1_A-08.csv')

Let's import well data from a data dump. Respectively: Flowline pressure, Flowline temperature and Bottom hole pressure (BHP).

In [ ]:
slug_1_df

In [ ]:
# plotting only 1 point per minutes to shorten process time
reduced_df1 = slug_1_df[::60]
reduced_df1

In [ ]:
fig, ts = plt.subplots(3,figsize=(10,10),sharex = True)

fig.suptitle('Slug dataset #1')
ts[2].set_xlabel('time')
ts[0].set_ylabel('Flowline P (Bar)')
ts[1].set_ylabel('Flowline T (deg)')
ts[2].set_ylabel('BHP (Bar)')

degrees = 70
plt.xticks(rotation=degrees)

ts[0].plot(reduced_df1[reduced_df1.columns[0]], reduced_df1["Flowline Pressure"])
ts[1].plot(reduced_df1[reduced_df1.columns[0]], reduced_df1["Flowline Temperature"])
ts[2].plot(reduced_df1[reduced_df1.columns[0]], reduced_df1["BHP"])


Is the steady state production in the timeseries a true white noise?
Taking a 'random' steady state interval for analysis.

In [ ]:
start_date = '2020-04-13 12:00:00'
end_date = '2020-04-13 18:00:00'
mask_ss = (slug_1_df[slug_1_df.columns[0]] > start_date) & (slug_1_df[slug_1_df.columns[0]] <= end_date)

steady_state = slug_1_df.loc[mask_ss]

fig_ss, ts_ss = plt.subplots(3,figsize=(10,10),sharex = True)

fig_ss.suptitle('Steady State Flow')
ts_ss[2].set_xlabel('time')
ts_ss[0].set_ylabel('Flowline P (Bar)')
ts_ss[1].set_ylabel('Flowline T (deg)')
ts_ss[2].set_ylabel('BHP (Bar)')

degrees = 70
plt.xticks(rotation=degrees)

ts_ss[0].plot(steady_state.index, steady_state["Flowline Pressure"])
ts_ss[1].plot(steady_state.index, steady_state["Flowline Temperature"])
ts_ss[2].plot(steady_state.index, steady_state["BHP"])

print('length of dataset', len(steady_state))

In [ ]:
fig_ss_analysis, data = plt.subplots(1, 3, figsize=(17, 3))

freq_units = 0.1/len(steady_state)
print('freq units: %(units)4E Hz' %{"units":(freq_units)})

fig_ss_analysis.suptitle('steady state BHP stats')
data[0].set_xlabel('time (10s)')
data[0].set_ylabel('BHP')
data[1].set_xlabel('freq (%(units).2E Hz)' %{"units":(freq_units)})
data[1].set_ylabel('power spectrum')
data[2].set_xlabel('BHP')
data[2].set_ylabel('count')

data[1].set_xlim(2,175)
data[1].set_ylim(0,2160)

ss_BHP = steady_state['BHP']

fft_ss = np.fft.rfft(ss_BHP)
fft_ss_abs = np.abs(fft_ss)
power_spectrum_ss = np.square(fft_ss_abs)


data[0].plot(ss_BHP)
data[1].plot(power_spectrum_ss)
ss_BHP.hist()


In [ ]:
f, Pxx_den = signal.periodogram(ss_BHP, 100)
plt.semilogy(f, Pxx_den)
plt.ylim([1e-10, 1e0])
plt.xlabel('frequency [Hz]')
plt.ylabel('PSD [V**2/Hz]')
plt.show()

In [ ]:
autocorrelation_plot(ss_BHP)

In [ ]:
qqplot(ss_BHP, line='s')

In [ ]:
# normality test
result = anderson(ss_BHP)
print('Statistic: %.3f' % result.statistic)
p = 0
for i in range(len(result.critical_values)):
	sl, cv = result.significance_level[i], result.critical_values[i]
	if result.statistic < result.critical_values[i]:
		print('%.3f: %.3f, data looks normal (fail to reject H0)' % (sl, cv))
	else:
		print('%.3f: %.3f, data does not look normal (reject H0)' % (sl, cv))

In [ ]:
import statsmodels.api as sm

fig = plt.figure(figsize=(12,15))
ax1 = fig.add_subplot(411)
fig = sm.graphics.tsa.plot_acf(ss_BHP.values.squeeze(), lags=500, ax=ax1)
ax2 = fig.add_subplot(412)
fig = sm.graphics.tsa.plot_acf(ss_BHP.values.squeeze(), lags=120, ax=ax2)
ax3 = fig.add_subplot(413)
fig = sm.graphics.tsa.plot_pacf(ss_BHP, lags=500, ax=ax3)
ax4 = fig.add_subplot(414)
fig = sm.graphics.tsa.plot_pacf(ss_BHP, lags=40, ax=ax4)

Well, the BHP data at steady state are not really normally distributed... How does the persisitent homology looks like?

We know when a slugging / irregulaf flow event occurred, so the dataframe is trimmed to a window of a manageable size centered around the event.

In [ ]:
max_time_delay = 200 
max_embedding_dimension = 10
stride = 10

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    ss_BHP, max_time_delay, max_embedding_dimension, stride=stride
    )
print('length of signal to analyze', int(len(ss_BHP)/stride))
print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")

embedding_dimension = optimal_embedding_dimension
embedding_time_delay = optimal_time_delay
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_ss_BHP = embedder.fit_transform(ss_BHP)

pca = PCA(n_components=3)
y_ss_BHP_pca = pca.fit_transform(y_ss_BHP)

plot_point_cloud(y_ss_BHP_pca)



In [ ]:
y_ss_BHP.shape

"y_ss_BHP" is the 5-dimensional array for steady state flow (i.e. normal operation). It is not alike Gaussian noise but from the PCA looks pretty spherical. 

In [ ]:
print('ss_BHP', ss_BHP.size)
ss_BHP_small = ss_BHP.head(1000)
print('ss_BHP_small', ss_BHP_small.size)
print('white_noise_gen', len(white_noise_gen))

# going down to a smaller array to reduce computational cost

Getting a bit more of details about the choice of the optimal time delay for the embedding... 

In [ ]:
y_ss_BHP

In [ ]:
import os
if not os.path.exists("images"):
    os.mkdir("images")
figure.write_image("images/point_cloud_steady.svg", width=2.1*300, height=2.1*300, scale=1)

In [ ]:
y_ss_BHP_reshaped = y_ss_BHP.reshape(1, *y_ss_BHP.shape)
PH_BHP =  VRP.fit_transform(y_ss_BHP_reshaped)
VRP.plot(PH_BHP)

In [ ]:
# write file for making figures  
file1 = open("SS_Pers_H0.dat", "w")
file2 = open("SS_Pers_H1.dat", "w")
for i in PH_BHP[0,:]:
    if i[2] == 0:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file1.writelines(tmp)
    elif i[2] == 1:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file2.writelines(tmp)
file1.close()
file2.close()

In [ ]:
PE_BHP = PersistenceEntropy()
features = PE_BHP.fit_transform(PH_BHP)
features 

In [ ]:
PE_BHP_norm = PersistenceEntropy(normalize=True)
features_norm = PE_BHP_norm.fit_transform(PH_BHP)
features_norm 

In [ ]:
Betti_BHP = Betti.fit_transform_plot(PH_BHP)

In [ ]:
file1 = open("SS_Betti_H0.dat", "w")
file2 = open("SS_Betti_H1.dat", "w")
min_H0 = 0.0
max_H0 = 0.06557
delta_H0 = (max_H0 - min_H0)/len(Betti_BHP[0,0])
min_H1 = 0.02988
max_H1 = 0.18428
delta_H1 = (max_H1 - min_H1)/len(Betti_BHP[0,1])
counter = 0 
for i in Betti_BHP[0,0]:
    tmp = str(min_H0 +(counter*delta_H0)) + str(" ") + str(i) + "\n"
    file1.writelines(tmp)
    counter += 1
file1.close()
counter = 0
for i in Betti_BHP[0,1]:
    tmp = str(min_H1 +(counter*delta_H1)) + str(" ") + str(i) + "\n"
    file2.writelines(tmp)
    counter += 1
file2.close()

In [ ]:
PH_BHP.shape

In [ ]:
pers_H0 = []
pers_H1 = []
max_pers_H0 = []
max_pers_H1 = []

for PersDiag in PH_BHP: 
    for point in PersDiag:
        birth = point[0]
        death = point[1]
        dimension = point[2]
        persistence = abs(death - birth)
        pers_H0.extend([persistence] if dimension == 0 else [])
        pers_H1.extend([persistence] if dimension == 1 else [])
    max_pers_H0.append(np.amax(pers_H0))
    max_pers_H1.append(np.amax(pers_H1))

print('Maximum Persistence for BHP:')
print("Max Pers in homology dimension H_0", max_pers_H0 )
print("Max Pers in homology dimension H_1", max_pers_H1 )

# Severe Slugging Event 1

The timeseries has a clear anomaly at the timestamp '2020-04-14 02:40:00', which we know is riser-induces slugging.

In [ ]:
start_date = '2020-04-14 03:00:00'
end_date = '2020-04-14 05:30:00'
mask = (slug_1_df[slug_1_df.columns[0]] > start_date) & (slug_1_df[slug_1_df.columns[0]] <= end_date)

In [ ]:
slug_event_1 = slug_1_df.loc[mask]

fig, ts = plt.subplots(3,figsize=(10,10),sharex = True)

fig.suptitle('Slug Event 1')
ts[2].set_xlabel('time')
ts[0].set_ylabel('Flowline P (Bar)')
ts[1].set_ylabel('Flowline T (deg)')
ts[2].set_ylabel('BHP (Bar)')

degrees = 70
plt.xticks(rotation=degrees)

ts[0].plot(slug_event_1.index, slug_event_1["Flowline Pressure"])
ts[1].plot(slug_event_1.index, slug_event_1["Flowline Temperature"])
ts[2].plot(slug_event_1.index, slug_event_1["BHP"])

In [ ]:
point_cloud = []

for entry in slug_event_1.index:
    FLW_P = slug_event_1["Flowline Pressure"][entry]
    FLW_T = slug_event_1["Flowline Temperature"][entry]
    BH_P = slug_event_1["BHP"][entry]
    point_cloud.append([FLW_P, FLW_T, BH_P])
        
plot_point_cloud(np.asarray(point_cloud))

The three timeseries show similar features: a series of seven peaks with an apparent period of 70 time intervals
(70 x 10s = 11.7 min). For sake of testing we focus here on BHP and apply a Taken embedding. 

There has been extensive work on understanding the behavior of the underlying dynamical system given only a 
time series [1, 2]. The revolutionary work of Takens [3] extended by Sauer et al. [4] showed that, 
given most choices of parameters, the state-space of the dynamical system can be reconstructed through the 
Takens’embedding. Computationally, this arises
as the following procedure. Given a time series [x1,··· , xn], a
choice of dimension d and time lag τ give rise to a point cloud
$χ = {xi:= (xi, xi+τ ,··· ,(xi+(d−1)τ)} ⊂ R_d. $

Then the goal is
to analyze this point cloud, which really is a sampling of the
full state space, in a way that the dynamics can be understood.

In [ ]:
slug_signal = slug_event_1["BHP"]
print('length of signal to analyze', len(slug_signal))

max_time_delay = 200 
max_embedding_dimension = 20
stride = 1

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    slug_signal, max_time_delay, max_embedding_dimension, stride=stride
    )

print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")

In [ ]:
embedding_dimension = 8
embedding_time_delay = 42
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_slug_embedded = embedder.fit_transform(slug_signal)

pca = PCA(n_components=3)
y_slug_embedded_pca = pca.fit_transform(y_slug_embedded)

plot_point_cloud(y_slug_embedded_pca)

y_slug_embedded is the 8-dimensional array of the embedded slugging (anomaly) event

In [ ]:
max_time_delay = 125

X = slug_signal
mutual_information_list = []

for time_delay in range(1, max_time_delay + 1):
    mutual_information_list.append(mutual_information(X, time_delay, n_bins=12))
           
plt.plot(range(1,max_time_delay+1),mutual_information_list);
plt.xlabel('delay');
plt.ylabel('mutual information');

In [ ]:
homology_dimensions = (0, 1, 2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

y_slug_embedded_reshaped = y_slug_embedded.reshape(1, *y_slug_embedded.shape)
print(f"y_slug_embedded_reshaped.shape",y_slug_embedded_reshaped.shape)
print(f"y_slug_embedded.shape", y_slug_embedded.shape)

X_diagrams = VRP.fit_transform(y_slug_embedded_reshaped)
VRP.plot(X_diagrams)

In [ ]:
pers_H0 = []
pers_H1 = []
max_pers_H0 = []
max_pers_H1 = []

for PersDiag in X_diagrams: 
    for point in PersDiag:
        birth = point[0]
        death = point[1]
        dimension = point[2]
        persistence = abs(death - birth)
        pers_H0.extend([persistence] if dimension == 0 else [])
        pers_H1.extend([persistence] if dimension == 1 else [])
    max_pers_H0.append(np.amax(pers_H0))
    max_pers_H1.append(np.amax(pers_H1))

print('Maximum Persistence for BHP:')
print("Max Pers in homology dimension H_0", max_pers_H0 )
print("Max Pers in homology dimension H_1", max_pers_H1 )

In [ ]:
PE_slug = PersistenceEntropy()
features = PE_slug.fit_transform(X_diagrams)
features

In [ ]:
PE_slug_norm = PersistenceEntropy(normalize=True)
features_norm = PE_slug_norm.fit_transform(X_diagrams)
features_norm

The entropy for H$_0$ is just barely lower than that for white noise, but the one for H$_1$ is much lower (6.0 vs 7.4). This clearly shows that there is some signal in the timeseries representing normal operation.

Let's start establishing a series of sliding windows along the slugging timeseries for the BHP for which 
we compute the persistent entropy. This is to test the ability of the persistent entropy as an indicator
for anomaly detection.

First, we isolate the periodic slugging pattern in the BHP signal. 

In [ ]:
slug_signal = slug_event_1["BHP"]
slug_signal

In [ ]:
slug_signal_200 = slug_signal[200:650]

fig = px.line()
#for dim in range(X_persistence_entropy.shape[1]):
fig.add_scatter(y=slug_signal_200, name="signal")
fig.show()

In [ ]:
max_time_delay = 20 
max_embedding_dimension = 12
stride = 1

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    slug_signal_200, max_time_delay, max_embedding_dimension, stride=stride
    )
print('length of signal to analyze', int(len(slug_signal_200)/stride))
print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")
embedding_dimension = optimal_time_delay
embedding_time_delay = optimal_time_delay
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_slug_embedded = embedder.fit_transform(slug_signal_200)

pca = PCA(n_components=3)
y_slug_embedded_pca = pca.fit_transform(y_slug_embedded)

slug = plot_point_cloud(y_slug_embedded_pca)
slug

y_slug_embedded is the 11-dimensional array of the embedded slugging event

In [ ]:
if not os.path.exists("images"):
    os.mkdir("images")
slug.write_image("images/point_cloud_slugging.svg", width=2.1*300, height=2.1*300, scale=1)

In [ ]:
homology_dimensions = (0, 1)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

y_slug_embedded_reshaped = y_slug_embedded.reshape(1, *y_slug_embedded.shape)
X_diagrams = VRP.fit_transform(y_slug_embedded_reshaped)
VRP.plot(X_diagrams)

In [ ]:
# write file for making figures  
file1 = open("Slug_Pers_H0.dat", "w")
file2 = open("Slug_Pers_H1.dat", "w")
for i in X_diagrams[0,:]:
    if i[2] == 0:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file1.writelines(tmp)
    elif i[2] == 1:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file2.writelines(tmp)
file1.close()
file2.close()

In [ ]:
Betti_slug = Betti.fit_transform_plot(X_diagrams)

In [ ]:
def betti(diagrams):
    # read the persistence diagram  
    pers_H0 = []
    pers_H1 = []
    max_pers_H0 = []
    max_pers_H1 = []

    for PersDiag in diagrams: 
        highest_death = [0.0, 0.0]
        for point in PersDiag:
            birth = point[0]
            death = point[1]
            dimension = point[2]
            persistence = abs(death - birth)
            if death > highest_death[int(dimension)]:
                highest_death[int(dimension)] = death
            
    x = np.empty([2, 100])
    y = np.empty([2, 100])
    for dimension in range(2): 
        for i in range(100):
            x[dimension, :] = np.linspace(0,highest_death[int(dimension)],100)
            y[dimension, :] = np.zeros(100)    
    for PersDiag in diagrams: 
        for point in PersDiag:
            birth = point[0]
            death = point[1]
            dimension = int(point[2])
            persistence = abs(death - birth)
            for nbin in range(len(x[dimension, :])):
                if birth < x[dimension,nbin] and death > x[dimension,nbin]:
                    y[int(dimension),nbin] += 1
                
    betti_array = np.concatenate((x,y),axis=0)
    
    return betti_array

In [ ]:
betti(X_diagrams).shape


#X_diagrams

In [ ]:
file1 = open("Slug_BettiH0.dat", "w")
file2 = open("Slug_BettiH1.dat", "w")
min_H0 = 0.0
max_H0 = 0.82947
delta_H0 = (max_H0 - min_H0)/len(Betti_slug[0,0])
min_H1 = 0.82947
max_H1 = 5.4535
delta_H1 = (max_H1 - min_H1)/len(Betti_slug[0,1])
counter = 0 
for i in Betti_slug[0,0]:
    tmp = str(min_H0 +(counter*delta_H0)) + str(" ") + str(i) + "\n"
    file1.writelines(tmp)
    counter += 1
file1.close()
counter = 0
for i in Betti_slug[0,1]:
    tmp = str(min_H1 +(counter*delta_H1)) + str(" ") + str(i) + "\n"
    file2.writelines(tmp)
    counter += 1
file2.close()

In [ ]:
print("mean of Betti_H0 for normal flow", stats.mean(Betti_BHP[0,0]))
print("mean of Betti_H1 for normal flow", stats.mean(Betti_BHP[0,1]))
print("mean of Betti_H0 for slug flow", stats.mean(Betti_slug[0,0]))
print("mean of Betti_H1 for slug flow", stats.mean(Betti_slug[0,1]))

In [ ]:
features_norm = PE_slug_norm.fit_transform(X_diagrams)
features = PE_slug.fit_transform(X_diagrams)
features

In [ ]:
features_norm

# Deep Dive Slugging Event

In [ ]:
start_date = '2020-04-14 00:30:00'
end_date = '2020-04-14 08:00:00'
mask = (slug_1_df[slug_1_df.columns[0]] > start_date) & (slug_1_df[slug_1_df.columns[0]] <= end_date)

In [ ]:
slug_event_large = slug_1_df.loc[mask]

fig, ts = plt.subplots(3,figsize=(10,10),sharex = True)

fig.suptitle('Slug Event')
ts[2].set_xlabel('time')
ts[0].set_ylabel('Flowline P (Bar)')
ts[1].set_ylabel('Flowline T (deg)')
ts[2].set_ylabel('BHP (Bar)')

degrees = 70
plt.xticks(rotation=degrees)

ts[0].plot(slug_event_large.index, slug_event_large["Flowline Pressure"])
ts[1].plot(slug_event_large.index, slug_event_large["Flowline Temperature"])
ts[2].plot(slug_event_large.index, slug_event_large["BHP"])

The periodicity of the slugging event is of about 75 timesteps, so we start with a sliding window of 450. 
Note that this is not too far from the periodicity of the signal in normal operations, therefore this can give rise to a lot of noise. 

In [ ]:
slug_signal = slug_event_large["BHP"]

window_size = 450
window_stride = 1
SW = SlidingWindow(size=window_size, stride=window_stride)

X_windows = SW.fit_transform(slug_signal)

print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")

TE = TakensEmbedding(time_delay=optimal_time_delay, dimension=optimal_embedding_dimension, stride=stride)
X_embedded_BHP450 = TE.fit_transform(X_windows)

In [ ]:
slug_signal

In [ ]:
X_embedded_BHP450.shape

In [ ]:
pca = PCA(n_components=3)
homology_dimensions = (0, 1)
persistence = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
entropy = PersistenceEntropy(normalize=True)

steps = [
         ("pca", batch_pca),
         ("persistence", persistence),
         ("entropy", entropy)
        ]
topological_transfomer = Pipeline(steps)

In [ ]:
X_persistence_entropy = topological_transfomer.fit_transform(X_embedded_BHP450)

fig = px.line(title='E_norm for BHP, indexed by sliding window number, window of 450 timesteps')
for dim in range(X_persistence_entropy.shape[1]):
    fig.add_scatter(y=X_persistence_entropy[:, dim], name=f"PE in homology dimension {dim}")
fig.show()

In [ ]:
pca = PCA(n_components=3)
homology_dimensions = (0, 1)
persistence = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
entropy = PersistenceEntropy()

steps = [
         ("pca", batch_pca),
         ("persistence", persistence),
         ("entropy", entropy)
        ]
topological_transfomer_notNorm = Pipeline(steps)

In [ ]:
X_persistence_entropy = topological_transfomer_notNorm.fit_transform(X_embedded_BHP450)

fig = px.line(title='E for BHP, indexed by sliding window number, window of 450 timesteps')
for dim in range(X_persistence_entropy.shape[1]):
    fig.add_scatter(y=X_persistence_entropy[:, dim], name=f"PE in homology dimension {dim}")
fig.show()

In [ ]:
slug_signal = slug_event_large["BHP"]

window_size = 450
window_stride = 2
SW = SlidingWindow(size=window_size, stride=window_stride)

X_windows = SW.fit_transform(slug_signal)

TE = TakensEmbedding(time_delay=14, dimension=11, stride=stride)
X_embedded_BHP450 = TE.fit_transform(X_windows)

In [ ]:
X_diagrams_BHP450 = VRP.fit_transform(X_embedded_BHP450)

In [ ]:
pers_H0 = []
pers_H1 = []
max_pers_H0 = []
max_pers_H1 = []

for PersDiag in X_diagrams_BHP450: 
    for point in PersDiag:
        birth = point[0]
        death = point[1]
        dimension = point[2]
        persistence = abs(death - birth)
        pers_H0.extend([persistence] if dimension == 0 else [])
        pers_H1.extend([persistence] if dimension == 1 else [])
    max_pers_H0.append(np.amax(pers_H0))
    max_pers_H1.append(np.amax(pers_H1))
    pers_H0 = []
    pers_H1 = []
           
fig = px.line(title='Maximum Persistence for BHP, indexed by sliding window number, window of 450 timesteps')
fig.add_scatter(y=max_pers_H0, name="Max Pers in homology dimension H_0")
fig.add_scatter(y=max_pers_H1, name="Max Pers in homology dimension H_1")
fig.show() 

In [ ]:
p_W = 2
PD = PairwiseDistance(metric='wasserstein',
                      metric_params={'p': p_W, 'delta': 0.1},
                      order=None)

#X_diagrams_BHP450 = persistence.fit_transform(X_embedded_BHP450)
X200_distance_W = PD.fit_transform(X_diagrams_BHP450)
plot_heatmap(X450_distance_W[:, :, 0], colorscale='blues')

In [ ]:
slug_signal = slug_event_large["BHP"]

window_size = 250
window_stride = 5
SW = SlidingWindow(size=window_size, stride=window_stride)
X_windows = SW.fit_transform(slug_signal)

max_time_delay = 12
max_embedding_dimension = 14
stride = 1

homology_dimensions = (0, 1)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
pca = PCA(n_components=3)
PE_slug = PersistenceEntropy()
PE_norm = PersistenceEntropy(normalize=True)

entropies = []
norm_entropies = []
parameters = []
diagrams = []
point_clouds = []
point_clouds_pca = []

for win in range(X_windows.shape[0]):
    optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
        X_windows[win], max_time_delay, max_embedding_dimension, stride=stride
        )
    if optimal_embedding_dimension < 3:
        optimal_embedding_dimension = 3

    parameters.append(np.column_stack((win, optimal_embedding_dimension, optimal_time_delay)))
    
    embedder = SingleTakensEmbedding(
        parameters_type="fixed", n_jobs=6, time_delay=optimal_time_delay, dimension=optimal_embedding_dimension, stride=stride
        )

    y_slug_embedded = embedder.fit_transform(X_windows[win])
    y_slug_embedded_reshaped = y_slug_embedded.reshape(1, *y_slug_embedded.shape)

    y_slug_embedded_pca = pca.fit_transform(y_slug_embedded)

    X_diagram = VRP.fit_transform(y_slug_embedded_reshaped)
    features = PE_slug.fit_transform(X_diagram)
    features_norm = PE_norm.fit_transform(X_diagram)

    point_clouds_pca.append(y_slug_embedded_pca)
    point_clouds.append(y_slug_embedded)
    entropies.append(features)
    norm_entropies.append(features_norm)
    diagrams.append(X_diagram)

In [ ]:
entropy_H0 = []
entropy_H1 = []
norm_entropy_H0 = []
norm_entropy_H1 = []

for i in entropies[:]:
    entropy_H0.append(i[0][0])
    entropy_H1.append(i[0][1])
    
for i in norm_entropies[:]:
    norm_entropy_H0.append(i[0][0])
    norm_entropy_H1.append(i[0][1])


In [ ]:
fig = px.line(title='E_pers for BHP, indexed by sliding window number, window of 250 timesteps')
fig.add_scatter(y=entropy_H0, name="PE in homology dimension H_0")
fig.add_scatter(y=entropy_H1, name="PE in homology dimension H_1")
fig.show()

In [ ]:
fig = px.line(title='E_norm for BHP, indexed by sliding window number, window of 250 timesteps')
fig.add_scatter(y=norm_entropy_H0, name="PE in homology dimension H_0")
fig.add_scatter(y=norm_entropy_H1, name="PE in homology dimension H_1")
fig.show()

In [ ]:
slug_signal = slug_event_large["BHP"]

window_size = 800
window_stride = 20
SW = SlidingWindow(size=window_size, stride=window_stride)

X_windows = SW.fit_transform(slug_signal)

TE = TakensEmbedding(time_delay=42, dimension=8, stride=stride)
X_embedded_BHP800 = TE.fit_transform(X_windows)

In [ ]:
X_windows.shape

In [ ]:
X_diagrams_BHP800 = VRP.fit_transform(X_embedded_BHP800)

In [ ]:
pers_H0 = []
pers_H1 = []
max_pers_H0 = []
max_pers_H1 = []

for PersDiag in X_diagrams_BHP800: 
    for point in PersDiag:
        birth = point[0]
        death = point[1]
        dimension = point[2]
        persistence = abs(death - birth)
        pers_H0.extend([persistence] if dimension == 0 else [])
        pers_H1.extend([persistence] if dimension == 1 else [])
    max_pers_H0.append(np.amax(pers_H0))
    max_pers_H1.append(np.amax(pers_H1))
    pers_H0 = []
    pers_H1 = []
    
# write file for making figures  
file1 = open("Max_Pers_H0.dat", "w")
file2 = open("Max_Pers_H1.dat", "w")
for i in max_pers_H0:
    tmp = str(i) + "\n"
    file1.writelines(tmp)
for i in max_pers_H1:
    tmp = str(i) + "\n"
    file2.writelines(tmp)
file1.close()
file2.close()
           
fig = px.line(title='Maximum Persistence for BHP, indexed by sliding window number, window of 800 timesteps')
fig.add_scatter(y=max_pers_H0, name="Max Pers in homology dimension H_0")
fig.add_scatter(y=max_pers_H1, name="Max Pers in homology dimension H_1")
fig.show() 

In [ ]:
PE = PersistenceEntropy()
X_persistence_entropy = PE.fit_transform(X_diagrams_BHP800)

file1 = open("Entropy_H0.dat", "w")
file2 = open("Entropy_H1.dat", "w")
for i in X_persistence_entropy[:, 0]:
    tmp = str(i) + "\n"
    file1.writelines(tmp)
for i in X_persistence_entropy[:, 1]:
    tmp = str(i) + "\n"
    file2.writelines(tmp)
file1.close()
file2.close()

fig = px.line(title='E_pers for BHP, indexed by sliding window number, window of 800 timesteps')
for dim in range(X_persistence_entropy.shape[1]):
    fig.add_scatter(y=X_persistence_entropy[:, dim], name=f"PE in homology dimension {dim}")
fig.show()

In [ ]:
PE_norm = PersistenceEntropy(normalize=True)
X_persistence_entropy_norm = PE_norm.fit_transform(X_diagrams_BHP800)

file1 = open("NormEntropy_H0.dat", "w")
file2 = open("NormEntropy_H1.dat", "w")
for i in X_persistence_entropy_norm[:, 0]:
    tmp = str(i) + "\n"
    file1.writelines(tmp)
for i in X_persistence_entropy_norm[:, 1]:
    tmp = str(i) + "\n"
    file2.writelines(tmp)
file1.close()
file2.close()

fig = px.line(title='nomalized E_pers for BHP, indexed by sliding window number, window of 800 timesteps')
for dim in range(X_persistence_entropy.shape[1]):
    fig.add_scatter(y=X_persistence_entropy_norm[:, dim], name=f"PE in homology dimension {dim}")
fig.show()

In [ ]:
Betti_BHP800 = Betti.fit_transform(X_diagrams_BHP800)
Betti_BHP800.shape

In [ ]:
figBetti = px.line()

for i in range(50):
    figBetti.add_scatter(y=(Betti_BHP800[i, 0, :]+5*i))
figBetti.show()


In [ ]:
figBetti = px.line()

for i in range(50,len((Betti_BHP800[:,0,0]))):
    figBetti.add_scatter(y=(Betti_BHP800[i, 0, :]-5*i))
figBetti.show()


In [ ]:
meanBettiH0 = []
medianBettiH0 = []
modeBettiH0 = []

for i in range(len(Betti_BHP800[:,0,0])):
    meanBettiH0.append(stats.mean(Betti_BHP800[i,0,:]))
    medianBettiH0.append(stats.median(Betti_BHP800[i,0,:]))
    modeBettiH0.append(stats.mode(Betti_BHP800[i,0,:]))
    
file1 = open("mean_Betti_H0.dat", "w")
file2 = open("median_Betti_H0.dat", "w")
for i in meanBettiH0:
    tmp = str(i) + "\n"
    file1.writelines(tmp)
for i in medianBettiH0:
    tmp = str(i) + "\n"
    file2.writelines(tmp)
file1.close()
file2.close()
    
figBetti = px.line()
figBetti.add_scatter(y=(meanBettiH0), name="Mean of Betti curve H_0")
figBetti.add_scatter(y=(medianBettiH0), name="Median of Betti curve H_0")
figBetti.add_scatter(y=(modeBettiH0),name="Mode of Betti curve H_0")
figBetti.show()


In [ ]:
figBetti = px.line(color_discrete_sequence=['gray', 'blue'])

for i in range(len(Betti_BHP800[:,0,0])):
#for i in range(50):
    figBetti.add_scatter(y=(Betti_BHP800[i, 1, :]+5*i))
figBetti.show()


In [ ]:
meanBettiH1 = []
medianBettiH1 = []
modeBettiH1 = []

for i in range(len(Betti_BHP800[:,1,0])):
    meanBettiH1.append(stats.mean(Betti_BHP800[i,1,:]))
    medianBettiH1.append(stats.median(Betti_BHP800[i,1,:]))
    modeBettiH1.append(stats.mode(Betti_BHP800[i,1,:]))

file1 = open("mean_Betti_H1.dat", "w")
file2 = open("median_Betti_H1.dat", "w")
for i in meanBettiH1:
    tmp = str(i) + "\n"
    file1.writelines(tmp)
for i in medianBettiH1:
    tmp = str(i) + "\n"
    file2.writelines(tmp)
file1.close()
file2.close()
    
figBetti = px.line()
figBetti.add_scatter(y=(meanBettiH1), name="Mean of Betti curve H_1")
figBetti.add_scatter(y=(medianBettiH1), name="Median of Betti curve H_1")
figBetti.add_scatter(y=(modeBettiH1),name="Mode of Betti curve H_1")
figBetti.show()


In [ ]:
p_W = 2
PD = PairwiseDistance(metric='wasserstein',
                      metric_params={'p': p_W, 'delta': 0.1},
                      order=None)

X800_distance_W = PD.fit_transform(X_diagrams_BHP800)

In [ ]:
plot_heatmap(X800_distance_W[:, :, 0], colorscale='blues')

In [ ]:
plot_heatmap(X800_distance_W[:, :, 1], colorscale='blues')

In [ ]:
# write file for making figures  
file1 = open("W2_H0_1x.dat", "w")
file2 = open("W2_H0_32x.dat", "w")
file3 = open("W2_H0_48x.dat", "w")

for i in X800_distance_W[:, 1, 0]:
    tmp = str(i) + "\n"
    file1.writelines(tmp)
for i in X800_distance_W[:, 32, 0]:
    tmp = str(i) + "\n"
    file2.writelines(tmp)
for i in X800_distance_W[:, 48, 0]:
    tmp = str(i) + "\n"
    file3.writelines(tmp)
file1.close()
file2.close()
file3.close()


fig1 = px.line()
fig1.add_scatter(y=X800_distance_W[:, 1, 0], name="distance #1-x")
fig1.add_scatter(y=X800_distance_W[:, 32, 0], name="distance #30-x")
fig1.add_scatter(y=X800_distance_W[:, 48, 0], name="distance #48-x")
fig1.show()

In [ ]:
# write file for making figures  
file1 = open("W2_H1_1x.dat", "w")
file2 = open("W2_H1_32x.dat", "w")
file3 = open("W2_H1_48x.dat", "w")

for i in X800_distance_W[:, 1, 1]:
    tmp = str(i) + "\n"
    file1.writelines(tmp)
for i in X800_distance_W[:, 32, 1]:
    tmp = str(i) + "\n"
    file2.writelines(tmp)
for i in X800_distance_W[:, 48, 1]:
    tmp = str(i) + "\n"
    file3.writelines(tmp)
file1.close()
file2.close()
file3.close()

fig1 = px.line()
fig1.add_scatter(y=X800_distance_W[:, 1, 1], name="distance #0-x")
fig1.add_scatter(y=X800_distance_W[:, 30, 1], name="distance #30-x")
fig1.add_scatter(y=X800_distance_W[:, 48, 1], name="distance #48-x")
fig1.show()

In [ ]:
p_L = 2
n_layers = 5
PD_L = PairwiseDistance(metric='landscape',
                      metric_params={'p': p_L, 'n_layers': n_layers, 'n_bins': 1000},
                      order=None)

X800_distance_L = PD_L.fit_transform(X_diagrams_BHP800)
plot_heatmap(X800_distance_L[:, :, 0], colorscale='reds')

In [ ]:
plot_heatmap(X800_distance_L[:, :, 1], colorscale='reds')

In [ ]:
PD_L = PairwiseDistance(metric='bottleneck')

X800_distance_B = PD_L.fit_transform(X_diagrams_BHP800)
plot_heatmap(X800_distance_B, colorscale='reds')

In [ ]:
p_L = 2
PD_img = PairwiseDistance(metric='persistence_image',
                      metric_params={'p': p_L, 'n_bins': 100},
                      order=None)

X800_distance_img = PD_L.fit_transform(X_diagrams_BHP800)


In [ ]:
plot_heatmap(X800_distance_img[:, :], colorscale='blues')

In [ ]:
window_number = 47
TE.plot(X_embedded_BHP800, sample=window_number)

In [ ]:
VRP.plot(X_diagrams_BHP800, sample=47)

In [ ]:
window_number = 10
TE.plot(X_embedded_BHP800, sample=window_number)

In [ ]:
VRP.plot(X_diagrams_BHP800, sample=10)

In [ ]:
PE_norm = PersistenceEntropy(normalize=True)
X_persistence_entropy_norm = PE_norm.fit_transform(X_diagrams_BHP900)

fig = px.line(title='nomalized E_pers for BHP, indexed by sliding window number, window of 900 timesteps')
for dim in range(X_persistence_entropy.shape[1]):
    fig.add_scatter(y=X_persistence_entropy_norm[:, dim], name=f"PE in homology dimension {dim}")
fig.show()

In [ ]:
PE_norm = PersistenceEntropy(normalize=True)
X_persistence_entropy_norm = PE_norm.fit_transform(X_diagrams_BHP900)

fig = px.line(title='nomalized E_pers for BHP, indexed by sliding window number, window of 900 timesteps')
for dim in range(X_persistence_entropy.shape[1]):
    if dim==2: 
        continue
    fig.add_scatter(y=X_persistence_entropy_norm[:, dim], name=f"PE in homology dimension {dim}")
    
fig.show()

In [ ]:
window_number = 100
TE.plot(X_embedded_BHP900, sample=window_number)

Testing the same approach but with a smaller time window, hoping to obtain a higher resolution.

In [ ]:
slug_signal = slug_event_large["BHP"]

window_size = 450
window_stride = 10
SW = SlidingWindow(size=window_size, stride=window_stride)

X_windows = SW.fit_transform(slug_signal)
optimal_time_delay = 42
optimal_embedding_dimension = 8
print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")
print(f"lenght of signal: {len(slug_signal)}")

TE = TakensEmbedding(time_delay=optimal_time_delay, dimension=optimal_embedding_dimension, stride=stride)
X_embedded_BHP450 = TE.fit_transform(X_windows)

In [ ]:
X_windows.shape

In [ ]:
len(X_windows[0])

Compute the persistent entropies for a sliding window approach with window_size = 450, where each window is 
processed at its optimal parameters.

In [ ]:
max_time_delay = 25
max_embedding_dimension = 15
stride = 1

homology_dimensions = (0, 1, 2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
pca = PCA(n_components=3)
PE_slug = PersistenceEntropy()
PE_norm = PersistenceEntropy(normalize=True)

entropies = []
norm_entropies = []
parameters = []
diagrams = []
point_clouds = []
point_clouds_pca = []

for win in range(X_windows.shape[0]):
    optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
        X_windows[win], max_time_delay, max_embedding_dimension, stride=stride
        )

    parameters.append(np.column_stack((win, optimal_embedding_dimension, optimal_time_delay)))
    
    embedder = SingleTakensEmbedding(
        parameters_type="search", n_jobs=6, time_delay=optimal_time_delay, dimension=optimal_embedding_dimension, stride=stride
        )

    y_slug_embedded = embedder.fit_transform(X_windows[win])
    y_slug_embedded_reshaped = y_slug_embedded.reshape(1, *y_slug_embedded.shape)

    y_slug_embedded_pca = pca.fit_transform(y_slug_embedded)

    X_diagram = VRP.fit_transform(y_slug_embedded_reshaped)
    features = PE_slug.fit_transform(X_diagram)
    features_norm = PE_norm.fit_transform(X_diagram)

    point_clouds_pca.append(y_slug_embedded_pca)
    point_clouds.append(y_slug_embedded)
    entropies.append(features)
    norm_entropies.append(features_norm)
    diagrams.append(X_diagram)

In [ ]:
entropy_H0 = []
entropy_H1 = []
entropy_H2 = []
norm_entropy_H0 = []
norm_entropy_H1 = []
norm_entropy_H2 = []

for i in entropies[:]:
    entropy_H0.append(i[0][0])
    entropy_H1.append(i[0][1])
    entropy_H2.append(i[0][2])
    
for i in norm_entropies[:]:
    norm_entropy_H0.append(i[0][0])
    norm_entropy_H1.append(i[0][1])
    norm_entropy_H2.append(i[0][2])

In [ ]:
fig = px.line(title='E_pers for BHP, indexed by sliding window number, window of 450 timesteps')
fig.add_scatter(y=entropy_H0, name=f"PE in homology dimension 0")
fig.add_scatter(y=entropy_H1, name=f"PE in homology dimension 1")
fig.add_scatter(y=entropy_H2, name=f"PE in homology dimension 2")
fig.show()

In [ ]:
fig = px.line(title='nomalized E_pers for BHP, indexed by sliding window number, window of 450 timesteps')
fig.add_scatter(y=norm_entropy_H0, name=f"PE in homology dimension 0")
fig.add_scatter(y=norm_entropy_H1, name=f"PE in homology dimension 1")
fig.add_scatter(y=norm_entropy_H2, name=f"PE in homology dimension 2")
layout_yaxis_range=[-10,49]
fig.show()

In [ ]:
plot_point_cloud(point_clouds_pca[101])

In [ ]:
VRP.plot(diagrams[101])

In [ ]:
plot_point_cloud(point_clouds_pca[56])

In [ ]:
VRP.plot(diagrams[56])

In [ ]:
plot_point_cloud(point_clouds_pca[121])

In [ ]:
VRP.plot(diagrams[121])

In [ ]:
PE_norm = PersistenceEntropy(normalize=True)
PE = PersistenceEntropy()
features_norm = PE_norm.fit_transform(diagrams[121])
features = PE.fit_transform(diagrams[121]) 
print('Entropy Normalized', features_norm)
print('Entropy', features)

In [ ]:
parameters[121]

In [ ]:
Betti.fit_transform_plot(diagrams[121])

In [ ]:
X_diagrams_BHP450 = VRP.fit_transform(X_embedded_BHP450)

In [ ]:
PE = PersistenceEntropy()
X_persistence_entropy = PE.fit_transform(X_diagrams_BHP450)

fig = px.line(title='Persistence entropies, indexed by sliding window number')
for dim in range(X_persistence_entropy.shape[1]):
    fig.add_scatter(y=X_persistence_entropy[:, dim], name=f"PE in homology dimension {dim}")
fig.show()

In [ ]:
PE_norm = PersistenceEntropy(normalize=True)
X_persistence_entropy_norm = PE_norm.fit_transform(X_diagrams_BHP450)

fig = px.line(title='nomalized E_pers for BHP, indexed by sliding window number, window of 450 timesteps')
for dim in range(X_persistence_entropy.shape[1]):
    if dim==2: 
        continue
    fig.add_scatter(y=X_persistence_entropy_norm[:, dim], name=f"PE in homology dimension {dim}")
    
fig.show()

In [ ]:
window_number = 77
TE.plot(X_embedded_BHP450, sample=window_number)

In [ ]:
TE.plot(X_embedded_BHP450, sample=126)

In [ ]:
SLH = Silhouette()
data_SLH = SLH.fit_transform_plot(X_diagrams_BHP450)

In [ ]:
dgms_steady = ripser(y_ss_BHP, maxdim=2)['dgms']
dgms_slug = ripser(y_slug_embedded, maxdim=2)['dgms']

In [ ]:
plot_diagrams(dgms_steady, plot_only=[1], ax=plt.subplot(121))
plt.title("Steady State $H_1$")

plot_diagrams(dgms_slug, plot_only=[1], ax=plt.subplot(122))
plt.title("Slug $H_1$")

In [ ]:
plot_diagrams(dgms, plot_only=[1], ax=plt.subplot(121))
plt.title("White Noise $H_1$")
plot_diagrams(dgms_slug, plot_only=[1], ax=plt.subplot(122))
plt.title("Slug $H_1$")

In [ ]:
from persim import bottleneck, bottleneck_matching, wasserstein, wasserstein_matching

#Bottleneck distance for H1 of steady state vs slugging 
d_bottle, b_matching = bottleneck(dgms_steady[1], dgms_slug[1], matching=True)
print('bottleneck distance steady state vs slugging', d_bottle)

bottleneck_matching(dgms_steady[1], dgms_slug[1], b_matching, labels=['Steady State $H_1$', 'Slug $H_1$'])
plt.title("Slug vs Steady State: Bottleneck Dist. {:.3f}".format(d_bottle))
plt.show()

In [ ]:
d_wasserstein, w_matching = wasserstein(dgms_steady[1], dgms_slug[1], matching=True)
print('wasserstein distance steady state vs slugging', d_wasserstein)
wasserstein_matching(dgms_steady[1], dgms_slug[1], w_matching, labels=['Steady State $H_1$', 'Slug $H_1$'])
plt.title("Slug vs Steady State: Wasserstein Dist. {:.3f}".format(d_wasserstein))
plt.show()

In [ ]:
#Bottleneck distance for H1 of white noise vs slugging
d_bottle = bottleneck(dgms[1], dgms_slug[1], matching=False)

d_bottle, b_matching = bottleneck(dgms[1], dgms_slug[1], matching=True)
print('bottleneck distance white noise vs slugging', d_bottle)

bottleneck_matching(dgms[1], dgms_slug[1], b_matching, labels=['Steady State $H_1$', 'Slug $H_1$'])
plt.title("Noise vs Steady State: Bottleneck Dist. {:.3f}".format(d_bottle))
plt.show()

In [ ]:
d_wasserstein, w_matching = wasserstein(dgms[1], dgms_slug[1], matching=True)
print('wasserstein distance white noise state vs slugging', d_wasserstein)
wasserstein_matching(dgms[1], dgms_slug[1], w_matching, labels=['Steady State $H_1$', 'Slug $H_1$'])
plt.title("Noise vs Slug: Wasserstein Dist. {:.3f}".format(d_wasserstein))
plt.show()

In [ ]:
#test 'self' distance (zero by definition)
d_bottle = bottleneck(dgms_slug[1], dgms_slug[1], matching=False)
d_wasserstein = wasserstein(dgms_slug[1], dgms_slug[1], matching=False)
print('bottleneck distance white noise vs slugging', d_bottle)
print('wasserstein distance white noise state vs slugging', d_wasserstein)

In [ ]:
X_diagrams 

In [ ]:
dgms[1]

In [ ]:
HK_slug = HK.fit_transform_plot(X_diagrams, homology_dimension_idx=1)
HK_steady = HK.fit_transform_plot(PH_BHP, homology_dimension_idx=1)
HK_wn = HK.fit_transform_plot(PerHom_wn, homology_dimension_idx = 1)

In [ ]:
PI_wn = PI.fit_transform_plot(PerHom_wn, homology_dimension_idx=1)
PI_steady = PI.fit_transform_plot(PH_BHP, homology_dimension_idx=1)
PI_slug = PI.fit_transform_plot(X_diagrams, homology_dimension_idx=1)

In [ ]:
from gtda.diagrams import BettiCurve 

Betti = BettiCurve()
Betti_wn = Betti.fit_transform_plot(PerHom_wn)
#Betti_slug = Betti.fit_transform_plot(X_diagrams)

In [ ]:
print('slug PH shape',X_diagrams.shape, 'white noise PH shape', PerHom_wn.shape) 
PerHom_wn_shrink = np.reshape(PerHom_wn, (1,807,3))
Diagrams = np.append(X_diagrams,PerHom_wn_shrink,axis=0)
Diagrams.shape

In [ ]:
from gtda.diagrams import PairwiseDistance
metrics = ['bottleneck', 'wasserstein', 'betti', 'landscape', 'silhouette', 'heat', 'image']

for metric in metrics:
    tmp = PairwiseDistance(metric=metric)
    print(metric, tmp)

In [ ]:
print(type(X_diagrams), X_diagrams.shape)
X_diagrams

In [ ]:
print(type(dgms[1]), dgms[1].shape)
dgms[1]

In [ ]:
X_diagrams_H1 = X_diagrams[:,:,1]
X_diagrams.shape
#X_diagrams


In [ ]:
X_diagrams[0,700:807,:]

In [ ]:
#X_diagrams_H1 = np.zeros(shape=(1,3))
X_diagrams_H1 = []

for i in X_diagrams:
    for y in i:
        if y[2] == 1:
            tmpy = [y[0],y[1]]
            #print(tmpy)
            #np.append(X_diagrams_H1, tmpy) 
            X_diagrams_H1.append(tmpy)

X_diagrams_H1 = np.asarray(X_diagrams_H1) 
X_diagrams_H1

In [ ]:
pimgr = PersistenceImager(pixel_size=1)
pimgr.fit(X_diagrams_H1)
#imgs = pimgr.transform(dgms[1])

fig_img, axs = plt.subplots(1, 3, figsize=(20,5))
pimgr.plot_diagram(X_diagrams_H1, skew=True, ax=axs[0])
axs[0].set_title('Diagram', fontsize=16)

pimgr.plot_image(pimgr.transform(X_diagrams_H1), ax=axs[1])
axs[1].set_title('Pixel Size: 1', fontsize=16)

pimgr.pixel_size = 0.01
pimgr.plot_image(pimgr.transform(X_diagrams_H1), ax=axs[2])
axs[2].set_title('Pixel Size: 0.01', fontsize=16)

plt.tight_layout()

In [ ]:
pippo = pd.read_csv('tmp.csv')

pippo